In [1]:
import cobra
import modelseedpy
import cobrakbase
from tqdm import tqdm
from cobrakbase.kbaseapi_cache import KBaseCache
config = cobra.Configuration()
#config.solver = 'cplex'
#config.solver = 'gurobi'

modelseedpy 0.4.0
cobrakbase 0.4.0


In [2]:
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [3]:
model_ids = set()
for i in kbase.list_workspace(186678):
    if i.type == 'KBaseFBA.FBAModel':
        if i.id.endswith('pyr.O2.gf'):
            model_ids.add(i.id)

In [4]:
data_methane = []
models = {}
for model_id in tqdm(model_ids):
    model = kbase.get_from_ws(model_id, 186678)
    models[model_id] = model
    if 'EX_cpd01024_e0' in model.reactions:
        model.objective = 'EX_cpd01024_e0'
        s = model.optimize()
        row_data = [
            model_id, s.objective_value, 
            [r.id for r in model.metabolites.cpd01024_c0.reactions]]
        data_methane.append(row_data)
        #print('!', row_data)

100%|██████████| 304/304 [09:44<00:00,  1.92s/it]


In [12]:
from cobra.core import Reaction
for model_id in models:
    model = models[model_id]
    if 'cpd25960_c0' in model.metabolites and 'EX_cpd25960_c0' not in model.reactions:
        ex = Reaction('EX_cpd25960_c0', 'exchange for MePn', '', -1000, 1000)
        ex.add_metabolites({model.metabolites.cpd25960_c0: -1})
        model.add_reactions([ex])
        print('!', model_id)

In [15]:
def _get_props(model, rxn_id, s):
    flux, genes = (None, None)
    if rxn_id in model.reactions:
        rxn = model.reactions.get_by_id(rxn_id)
        flux = s.fluxes[rxn_id]
        genes = {x.id for x in rxn.genes}
    return flux, genes

In [20]:
for model_id in models:
    model = models[model_id]
    
    if 'EX_cpd01024_e0' in model.reactions:
        model.objective = 'EX_cpd01024_e0'
        s = model.optimize()
        print(model_id, s.objective_value),
        print('\t', 'rxn26456_c0' in model.reactions, _get_props(model, 'rxn26456_c0', s))
        print('\t', 'rxn26457_c0' in model.reactions, _get_props(model, 'rxn26457_c0', s))
        print('\t', 'rxn43120_c0' in model.reactions, _get_props(model, 'rxn43120_c0', s))
        print()
    else:
        print(model_id, ['rxn26456_c0' in model.reactions, 'rxn26457_c0' in model.reactions, 'rxn43120_c0' in model.reactions], '-')
        print()

Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.pangenome.RAST.glm4ec.pyr.O2.gf 0.0
	 True (0.0, {'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1733', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1739', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1735', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1739_mRNA', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1734_mRNA', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1734', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1733_mRNA', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1735_mRNA'})
	 True (0.0, {'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1744_mRNA', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.1742_mRNA', 'Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.CDS.174

In [6]:
for o in data_methane:
    print(o)

['Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.98.pangenome.RAST.glm4ec.pyr.O2.gf', 0.0, ['rxn00843_c0', 'rxn10471_c0']]
['Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.17.pangenome.RAST.glm4ec.pyr.O2.gf', 100.0, ['rxn00843_c0', 'rxn10471_c0', 'rxn03127_c0', 'rxn26459_c0']]
['Salt_Pond_MetaG_R2A_A_H2O_MG_DASTool_bins_metabat.16.pangenome.RAST.glm4ec.pyr.O2.gf', 100.0, ['rxn00843_c0', 'rxn03127_c0', 'rxn43120_c0', 'rxn10471_c0']]
['Salt_Pond_MetaG_R2_A_D1_MG_DASTool_bins_concoct_out.84.pangenome.RAST.glm4ec.pyr.O2.gf', 100.0, ['rxn00843_c0', 'rxn10471_c0', 'rxn03127_c0']]
['Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.19.pangenome.RAST.glm4ec.pyr.O2.gf', 100.0, ['rxn03127_c0', 'rxn00843_c0', 'rxn10471_c0', 'rxn26459_c0']]
['Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.120.pangenome.RAST.glm4ec.pyr.O2.gf', 50.0, ['rxn00843_c0', 'rxn10471_c0', 'rxn03127_c0']]
['Salt_Pond_MetaGSF2_A_H2O_MG_DASTool_bins_metabat.28.pangenome.RAST.glm4ec.pyr.O2.gf', 0.0, ['rxn10471_c0'

{'rxn03127_c0', 'rxn10471_c0'}

100.0

In [19]:
model.metabolites.cpd01024_c0

Metabolite identifier,cpd01024_c0
Name,Methane [c0]
Memory address,0x7f4498391b10
Formula,CH4
Compartment,c0
In 2 reaction(s),"rxn10471_c0, rxn03127_c0"


In [20]:
model.reactions.rxn03127_c0

Reaction identifier,rxn03127_c0
Name,2-(Methylthio)ethanesulfonate:N-(7-thioheptanoyl)-3-O-phosphothreonine...
Memory address,0x7f4493816ad0
Stoichiometry,cpd02438_c0 + cpd02817_c0 --> cpd00067_c0 + cpd01024_c0 + cpd02935_c0 Methyl CoM [c0] + HTP [c0] --> H+ [c0] + Methane [c0] + CoM-S-S-CoB [c0]
GPR,
Lower bound,0
Upper bound,100


In [2]:
ws_reg = 155805
ws_enh = 186678

In [4]:
model_reg = kbase.get_from_ws('Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.37.contigs__.RAST.pyr.O2', ws_reg)
model_enh = kbase.get_from_ws('Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.37.pangenome.RAST.glm4ec.pyr.O2', ws_enh)

In [23]:
model_reg = kbase.get_from_ws('Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.57.contigs__.RAST.pyr.O2', ws_reg)
model_enh = kbase.get_from_ws('Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.57.pangenome.RAST.glm4ec.pyr.O2', ws_enh)

In [28]:
model_reg.template_ref

'155805/767/2'

In [29]:
model_enh.template_ref

'186678/12/1'

In [30]:
kbase.get_object_info(model_reg.template_ref).id

'template_v6gn_methylphosphonate_deg'

In [31]:
kbase.get_object_info(model_enh.template_ref).id

'template_v6gn_methylphosphonate_deg'

In [26]:
len(model_reg.genes), len(model_enh.genes)

(1431, 1505)

In [27]:
len(model_reg.reactions), len(model_enh.reactions)

(1078, 1100)

In [21]:
model_enh.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
cpd00007_e0,EX_cpd00007_e0,866.4,0,0.00%
cpd00013_e0,EX_cpd00013_e0,123.2,0,0.00%
cpd00028_e0,EX_cpd00028_e0,0.07008,34,0.02%
cpd00030_e0,EX_cpd00030_e0,0.07008,0,0.00%
cpd00034_e0,EX_cpd00034_e0,0.07008,0,0.00%
cpd00039_e0,EX_cpd00039_e0,12.3,6,0.71%
cpd00042_e0,EX_cpd00042_e0,9.139,10,0.88%
cpd00047_e0,EX_cpd00047_e0,11.37,1,0.11%
cpd00048_e0,EX_cpd00048_e0,0.07008,0,0.00%
cpd00051_e0,EX_cpd00051_e0,10.63,6,0.62%


In [15]:
for rxn_id in {x.id for x in model_enh.reactions} - {x.id for x in model_reg.reactions}:
    print(model_enh.reactions.get_by_id(rxn_id).build_reaction_string(True))

ADP [c0] + H+ [c0] + Nicotinate ribonucleotide [c0] <-- ATP [c0] + Nicotinate D-ribonucleoside [c0]
H2O [c0] + O2 [c0] + L-Lysine [c0] --> NH3 [c0] + H2O2 [c0] + 2-Oxo-6-aminocaproate [c0]
GTP [c0] + Mevalonic acid [c0] --> GDP [c0] + H+ [c0] + 5-phosphomevalonate [c0]
NADP [c0] + Phosphate [c0] + Glyceraldehyde3-phosphate [c0] <-- NADPH [c0] + H+ [c0] + 1,3-Bisphospho-D-glycerate [c0]
H2O [c0] + N-acetyl-LL-2,6-diaminopimelate [c0] --> Acetate [c0] + LL-2,6-Diaminopimelate [c0]
2 Isopentenyldiphosphate [c0] + Geranylgeranyl diphosphate [c0] --> 2 PPi [c0] + 2 H+ [c0] + all-trans-Hexaprenyl diphosphate [c0]
L-Glutamine [c0] + Glyceraldehyde3-phosphate [c0] + D-Ribulose5-phosphate [c0] --> 3 H2O [c0] + Phosphate [c0] + Pyridoxal phosphate [c0] + L-Glutamate [c0] + H+ [c0]
NAD [c0] + Glycolate [c0] <-- NADH [c0] + Glyoxalate [c0] + H+ [c0]
2 NADP [c0] + CoA [c0] + Mevalonic acid [c0] <-- 2 NADPH [c0] + 2 H+ [c0] + HMG-CoA [c0]
NADH [c0] + NADP [c0] + 2 H+ [e0] --> NAD [c0] + NADPH [c0] +

In [18]:
for rxn_id in {x.id for x in model_reg.reactions} - {x.id for x in model_enh.reactions}:
    print(rxn_id, model_reg.reactions.get_by_id(rxn_id).build_reaction_string(True))

rxn03907_c0 CTP [c0] + 2-C-methyl-D-erythritol4-phosphate [c0] --> PPi [c0] + 4--cytidine5-diphospho-2-C-methyl-D-erythritol [c0]
rxn01424_c0 H+ [c0] + Lipoamide [c0] + 2-Oxoadipate [c0] --> CO2 [c0] + S-Glutaryldihydrolipoamide [c0]
rxn00239_c0 ATP [c0] + H+ [c0] + GMP [c0] --> ADP [c0] + GDP [c0]
rxn08981_c0 H2O [c0] + ATP [c0] + PRPP [c0] + Niacin [c0] --> ADP [c0] + Phosphate [c0] + PPi [c0] + H+ [c0] + Nicotinate ribonucleotide [c0]
rxn02939_c0 NAD [c0] + 4-Phosphoerythronate [c0] --> NADH [c0] + H+ [c0] + 2-Oxo-3-hydroxy-4-phosphobutanoate [c0]
rxn00966_c0 Pyruvate [c0] + 4-Hydroxybenzoate [c0] <-- Chorismate [c0]
rxn03004_c0 10-Formyltetrahydrofolate [c0] + GAR [c0] --> H+ [c0] + Tetrahydrofolate [c0] + N-Formyl-GAR [c0]
rxn01873_c0 Glutaryl-CoA [c0] + Dihydrolipoamide [c0] <-- CoA [c0] + H+ [c0] + S-Glutaryldihydrolipoamide [c0]
rxn04413_c0 GTP [c0] + Adenosyl cobinamide [c0] --> GDP [c0] + H+ [c0] + Adenosyl cobinamide phosphate [c0]
rxn10094_c0 S-Adenosyl-L-methionine [c0] + 

In [22]:
for info in kbase.list_workspace(ws_reg):
    if info.type == 'KBaseFBA.FBAModel':
        print(info.id, info.type)

Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.50.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.19.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_A_D2_MG_DASTool_bins_concoct_out.19.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_B_H2O_MG_DASTool_bins_metabat.7.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_A_H2O_MG_DASTool_bins_maxbin.022.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_concoct_out.51.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.7.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_C_H2O_MG_DASTool_bins_metabat.29.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.32.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_metabat.18.contigs__.RAST.auxo.mdl KBaseFBA.FBAModel
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_co

In [ ]:
for info in kbase.list_workspace(ws_enh):
    if info.type == 'KBaseFBA.FBAModel':
        kbase.get_from_ws(info.id, ws_enh)
        print(info.id, info.type)